# Inline Filament Dryer — Optimization Study

Multi-objective optimization for inline filament dryer design.

Finds optimal `(chamber_length, chamber_temp, airflow_velocity)` by minimizing
a weighted cost of moisture remaining, chamber length, and energy usage.
Uses `dlib.find_min_global`.

## 1. Imports & Setup

In [ ]:
from __future__ import annotations

import time
from dataclasses import dataclass, field

import dlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from dryer_core.dryer import DryerConfig, DryingResult, FilamentConfig, simulate
from dryer_core.materials import MATERIALS

## 2. Configuration Dataclasses

In [2]:
@dataclass
class OptimizationConfig:
    """Configuration for multi-objective dryer optimization.

    Attributes:
        filament: Fixed filament configuration (material, diameter, moisture, flow_rate).
        bounds_length: Min/max chamber length [m].
        bounds_temp: Min/max chamber temperature [°C]. Upper bound is clamped
            to material.max_temp automatically.
        bounds_airflow: Min/max airflow velocity [m/s].
        weight_moisture: Cost weight for moisture remaining fraction.
        weight_length: Cost weight for normalized chamber length.
        weight_energy: Cost weight for energy proxy.
        target_moisture: If set, final moisture above this incurs a large penalty.
            Defaults to material.max_print_moisture when None.
        ambient_humidity: Ambient relative humidity (0–1).
        ambient_temp: Ambient temperature [°C].
    """

    filament: FilamentConfig
    bounds_length: tuple[float, float] = (0.1, 50.0)
    bounds_temp: tuple[float, float] = (40.0, 120.0)
    bounds_airflow: tuple[float, float] = (0.1, 10.0)
    weight_moisture: float = 1.0
    weight_length: float = 1.0
    weight_energy: float = 1.0
    target_moisture: float | None = None
    ambient_humidity: float = 0.50
    ambient_temp: float = 25.0

    def effective_target_moisture(self) -> float | None:
        """Return the active target moisture, falling back to material threshold."""
        if self.target_moisture is not None:
            return self.target_moisture
        return self.filament.material.max_print_moisture


@dataclass
class OptimizationResult:
    """Result of a dryer optimization run.

    Attributes:
        optimal_length: Best chamber length [m].
        optimal_temp: Best chamber temperature [°C].
        optimal_airflow: Best airflow velocity [m/s].
        optimal_cost: Total weighted cost at the optimum.
        cost_breakdown: Individual cost components (pre-weighting).
        simulation: Full DryingResult at the optimal point.
        config: The OptimizationConfig used.
        eval_history: List of (x, cost) tuples recorded during optimization.
        wall_time: Total wall-clock time for the optimization [s].
    """

    optimal_length: float
    optimal_temp: float
    optimal_airflow: float
    optimal_cost: float
    cost_breakdown: dict[str, float] = field(default_factory=dict)
    simulation: DryingResult | None = None
    config: OptimizationConfig | None = None
    eval_history: list[dict] = field(default_factory=list)
    wall_time: float = 0.0

    def summary(self) -> str:
        lines = [
            f"Optimal chamber length:  {self.optimal_length * 1e3:.0f} mm",
            f"Optimal chamber temp:    {self.optimal_temp:.1f} °C",
            f"Optimal airflow:         {self.optimal_airflow:.2f} m/s",
            "",
            f"Total cost:              {self.optimal_cost:.6f}",
        ]
        for name, value in self.cost_breakdown.items():
            lines.append(f"  {name:25s} {value:.6f}")
        if self.simulation:
            sim = self.simulation
            target = self.config.effective_target_moisture() if self.config else None
            lines += [
                "",
                f"Final moisture:          {sim.final_moisture * 100:.3f} wt%",
                f"Drying efficiency:       {sim.drying_efficiency * 100:.2f} %",
                f"Transit time:            {sim.transit_time:.2f} s",
                f"Fourier number:          {sim.fourier_number:.4e}",
            ]
            if target is not None:
                met = "YES" if sim.final_moisture <= target else "NO"
                lines.append(f"Print threshold:         {target * 100:.2f} wt% — met: {met}")
        lines.append(f"\nWall time:               {self.wall_time:.2f} s")
        lines.append(f"Function evaluations:    {len(self.eval_history)}")
        return "\n".join(lines)

## 3. Objective Function & Optimizer

In [3]:
def _objective(x: list[float], config: OptimizationConfig) -> float:
    """Scalar cost function for the optimizer.

    Cost components (each normalized to [0, 1]):
      - moisture: final_moisture / initial_moisture  (fraction remaining)
      - length:   chamber_length / max_length
      - energy:   (ΔT/ΔT_max) × (v/v_max)²
                   Heating power ∝ ΔT, fan power ∝ v².
    """
    chamber_length, chamber_temp, airflow_velocity = x

    dryer = DryerConfig(
        chamber_length=chamber_length,
        chamber_temp=chamber_temp,
        ambient_humidity=config.ambient_humidity,
        ambient_temp=config.ambient_temp,
        airflow_velocity=airflow_velocity,
    )

    result = simulate(dryer, config.filament, N=30, t_eval_count=50)

    # --- Normalized cost components ---
    cost_moisture = result.final_moisture / result.initial_moisture

    cost_length = chamber_length / config.bounds_length[1]

    delta_t = chamber_temp - config.ambient_temp
    delta_t_max = config.bounds_temp[1] - config.ambient_temp
    v_norm = airflow_velocity / config.bounds_airflow[1]
    cost_energy = (delta_t / delta_t_max) * v_norm**2

    cost = (
        config.weight_moisture * cost_moisture
        + config.weight_length * cost_length
        + config.weight_energy * cost_energy
    )

    # Soft penalty for target moisture constraint (uses material threshold by default)
    target = config.effective_target_moisture()
    if target is not None and result.final_moisture > target:
        overshoot = result.final_moisture - target
        cost += 100.0 * overshoot

    return cost

In [4]:
def optimize(config: OptimizationConfig, num_evaluations: int = 50) -> OptimizationResult:
    """Run multi-objective optimization to find best dryer parameters.

    Uses dlib.find_min_global (LIPO+TR).
    Decision variables: [chamber_length, chamber_temp, airflow_velocity].
    Records every evaluation for convergence analysis.

    Args:
        config: Optimization configuration.
        num_evaluations: Exact number of objective function calls dlib will make.
    """
    # Clamp temperature upper bound to material safe limit
    temp_upper = min(config.bounds_temp[1], config.filament.material.max_temp)

    lower_bounds = [config.bounds_length[0], config.bounds_temp[0], config.bounds_airflow[0]]
    upper_bounds = [config.bounds_length[1], temp_upper, config.bounds_airflow[1]]

    # Track every evaluation
    eval_history: list[dict] = []

    def obj(chamber_length, chamber_temp, airflow_velocity):
        t0 = time.perf_counter()
        cost = _objective([chamber_length, chamber_temp, airflow_velocity], config)
        dt = time.perf_counter() - t0
        eval_history.append({
            "length": chamber_length,
            "temp": chamber_temp,
            "airflow": airflow_velocity,
            "cost": cost,
            "eval_time": dt,
        })
        return cost

    t_start = time.perf_counter()
    opt_x, opt_y = dlib.find_min_global(
        obj,
        lower_bounds,
        upper_bounds,
        num_evaluations,
    )
    wall_time = time.perf_counter() - t_start

    opt_length, opt_temp, opt_airflow = opt_x

    # Re-simulate at optimum with full resolution for detailed result
    dryer = DryerConfig(
        chamber_length=opt_length,
        chamber_temp=opt_temp,
        ambient_humidity=config.ambient_humidity,
        ambient_temp=config.ambient_temp,
        airflow_velocity=opt_airflow,
    )
    sim = simulate(dryer, config.filament)

    # Recompute cost breakdown for reporting
    cost_moisture = sim.final_moisture / sim.initial_moisture
    cost_length = opt_length / config.bounds_length[1]
    delta_t = opt_temp - config.ambient_temp
    delta_t_max = config.bounds_temp[1] - config.ambient_temp
    v_norm = opt_airflow / config.bounds_airflow[1]
    cost_energy = (delta_t / delta_t_max) * v_norm**2

    breakdown = {
        f"moisture  (×{config.weight_moisture})": config.weight_moisture * cost_moisture,
        f"length   (×{config.weight_length})": config.weight_length * cost_length,
        f"energy   (×{config.weight_energy})": config.weight_energy * cost_energy,
    }

    return OptimizationResult(
        optimal_length=opt_length,
        optimal_temp=opt_temp,
        optimal_airflow=opt_airflow,
        optimal_cost=opt_y,
        cost_breakdown=breakdown,
        simulation=sim,
        config=config,
        eval_history=eval_history,
        wall_time=wall_time,
    )

## 4. Calibration: How Many Evaluations?

`dlib.find_min_global` takes an exact `num_function_calls` — unlike iterative
optimizers it doesn't have a convergence criterion. Sweep evaluation budgets
to find where the optimal cost stabilises.

In [5]:
filament = FilamentConfig(material=MATERIALS["PA6"], initial_moisture=0.03, flow_rate=8.0)

config = OptimizationConfig(filament=filament)

budgets = [5, 10, 20, 40, 80, 160, 320]
budget_results = []

for b in budgets:
    r = optimize(config, num_evaluations=b)
    budget_results.append({
        "num_evals": b,
        "optimal_cost": r.optimal_cost,
        "wall_time": r.wall_time,
        "length_mm": r.optimal_length * 1e3,
        "temp_C": r.optimal_temp,
        "airflow_ms": r.optimal_airflow,
        "final_moisture_pct": r.simulation.final_moisture * 100,
    })
    print(f"  n={b:4d}  cost={r.optimal_cost:.6f}  L={r.optimal_length*1e3:.0f}mm  T={r.optimal_temp:.1f}°C  v={r.optimal_airflow:.2f}m/s  ({r.wall_time:.1f}s)")

df_budget = pd.DataFrame(budget_results)

NameError: name 'dlib' is not defined

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Cost vs budget
ax = axes[0]
ax.plot(df_budget["num_evals"], df_budget["optimal_cost"], "o-", lw=2)
ax.set_xlabel("num_function_calls")
ax.set_ylabel("Optimal cost")
ax.set_title("Cost vs evaluation budget")
ax.grid(True, ls=":", alpha=0.4)

# Wall time vs budget
ax = axes[1]
ax.plot(df_budget["num_evals"], df_budget["wall_time"], "s-", color="coral", lw=2)
ax.set_xlabel("num_function_calls")
ax.set_ylabel("Wall time [s]")
ax.set_title("Wall time vs evaluation budget")
ax.grid(True, ls=":", alpha=0.4)

# Decision variables vs budget
ax = axes[2]
ax.plot(df_budget["num_evals"], df_budget["length_mm"], "o-", label="Length [mm]")
ax.plot(df_budget["num_evals"], df_budget["temp_C"], "s-", label="Temp [°C]")
ax.plot(df_budget["num_evals"], df_budget["airflow_ms"] * 100, "^-", label="Airflow [cm/s]")
ax.set_xlabel("num_function_calls")
ax.set_ylabel("Value")
ax.set_title("Optimal parameters vs budget")
ax.legend(fontsize=8)
ax.grid(True, ls=":", alpha=0.4)

plt.tight_layout()

In [ ]:
# Find the first budget where cost is within 1% of the best result
best_cost = df_budget["optimal_cost"].min()
threshold = best_cost * 1.01

converged = df_budget[df_budget["optimal_cost"] <= threshold].iloc[0]
converged_n = int(converged["num_evals"])

# Apply 2× safety margin (other materials may be harder)
NUM_EVALUATIONS = converged_n * 2

print(f"Best cost achieved:  {best_cost:.6f} (at n={df_budget.loc[df_budget['optimal_cost'].idxmin(), 'num_evals']:.0f})")
print(f"Within 1% threshold: {threshold:.6f}")
print(f"First budget to converge: n={converged_n}")
print(f"With 2× safety margin:   NUM_EVALUATIONS = {NUM_EVALUATIONS}")

## 5. Run Optimization

Run with the calibrated evaluation budget.

In [ ]:
result = optimize(config, num_evaluations=NUM_EVALUATIONS)
print(result.summary())

## 6. Baseline Comparison

Compare the optimized dryer against a naive 500 mm / 80 °C / 1 m/s baseline.

In [ ]:
baseline = simulate(
    DryerConfig(chamber_length=0.5, chamber_temp=80.0),
    filament,
)

comparison = pd.DataFrame({
    "Parameter": [
        "Chamber length [mm]",
        "Chamber temp [°C]",
        "Airflow [m/s]",
        "Final moisture [wt%]",
        "Drying efficiency [%]",
        "Transit time [s]",
    ],
    "Baseline": [
        500.0,
        80.0,
        1.0,
        f"{baseline.final_moisture * 100:.3f}",
        f"{baseline.drying_efficiency * 100:.2f}",
        f"{baseline.transit_time:.2f}",
    ],
    "Optimized": [
        f"{result.optimal_length * 1e3:.0f}",
        f"{result.optimal_temp:.1f}",
        f"{result.optimal_airflow:.2f}",
        f"{result.simulation.final_moisture * 100:.3f}",
        f"{result.simulation.drying_efficiency * 100:.2f}",
        f"{result.simulation.transit_time:.2f}",
    ],
})
comparison

## 7. Optimizer Convergence

How the best-so-far cost evolves over function evaluations — shows whether
the optimizer has converged or would benefit from more evaluations.

In [ ]:
df_hist = pd.DataFrame(result.eval_history)
df_hist["eval_num"] = range(1, len(df_hist) + 1)
df_hist["best_so_far"] = df_hist["cost"].cummin()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: convergence curve
ax = axes[0]
ax.scatter(df_hist["eval_num"], df_hist["cost"], s=8, alpha=0.3, label="Each evaluation")
ax.plot(df_hist["eval_num"], df_hist["best_so_far"], "r-", lw=2, label="Best so far")
ax.set_xlabel("Function evaluation #")
ax.set_ylabel("Cost")
ax.set_title("Optimizer convergence")
ax.legend()
ax.grid(True, ls=":", alpha=0.4)

# Right: log-scale improvement rate
ax = axes[1]
improvement = df_hist["best_so_far"].iloc[0] - df_hist["best_so_far"]
# Avoid log(0) by masking zero improvements
mask = improvement > 0
ax.plot(df_hist["eval_num"][mask], improvement[mask], "g-", lw=1.5)
ax.set_xlabel("Function evaluation #")
ax.set_ylabel("Cumulative improvement from initial best")
ax.set_yscale("log")
ax.set_title("Improvement rate (log scale)")
ax.grid(True, ls=":", alpha=0.4)

plt.tight_layout()

## 8. Per-Evaluation Timing

Distribution of wall-clock time per objective function call —
useful for estimating total optimization budget and identifying outliers.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

eval_times_ms = df_hist["eval_time"] * 1000

# Left: histogram
ax = axes[0]
ax.hist(eval_times_ms, bins=40, edgecolor="k", alpha=0.7)
ax.axvline(eval_times_ms.median(), color="r", ls="--", label=f"Median: {eval_times_ms.median():.1f} ms")
ax.axvline(eval_times_ms.mean(), color="orange", ls="--", label=f"Mean: {eval_times_ms.mean():.1f} ms")
ax.set_xlabel("Evaluation time [ms]")
ax.set_ylabel("Count")
ax.set_title("Objective function call duration")
ax.legend(fontsize=9)

# Right: time per eval over sequence (detect ramp-up / cache effects)
ax = axes[1]
ax.plot(df_hist["eval_num"], eval_times_ms, lw=0.8, alpha=0.6)
ax.axhline(eval_times_ms.median(), color="r", ls="--", lw=0.8)
ax.set_xlabel("Function evaluation #")
ax.set_ylabel("Time [ms]")
ax.set_title("Per-evaluation timing over sequence")
ax.grid(True, ls=":", alpha=0.4)

plt.tight_layout()

print(f"Total wall time:  {result.wall_time:.2f} s")
print(f"Mean eval time:   {eval_times_ms.mean():.2f} ms")
print(f"Median eval time: {eval_times_ms.median():.2f} ms")
print(f"Min / Max:        {eval_times_ms.min():.2f} / {eval_times_ms.max():.2f} ms")

## 9. Cost Component Breakdown

How the three weighted cost components contribute at the optimum,
and how they evolved during the search.

In [ ]:
# Recompute cost components for every evaluation point
cost_components = {"moisture": [], "length": [], "energy": []}
for row in result.eval_history:
    dryer_i = DryerConfig(
        chamber_length=row["length"],
        chamber_temp=row["temp"],
        ambient_humidity=config.ambient_humidity,
        ambient_temp=config.ambient_temp,
        airflow_velocity=row["airflow"],
    )
    sim_i = simulate(dryer_i, config.filament, N=30, t_eval_count=50)

    cost_components["moisture"].append(
        config.weight_moisture * sim_i.final_moisture / sim_i.initial_moisture
    )
    cost_components["length"].append(
        config.weight_length * row["length"] / config.bounds_length[1]
    )
    delta_t = row["temp"] - config.ambient_temp
    delta_t_max = config.bounds_temp[1] - config.ambient_temp
    v_norm = row["airflow"] / config.bounds_airflow[1]
    cost_components["energy"].append(
        config.weight_energy * (delta_t / delta_t_max) * v_norm**2
    )

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: pie chart at optimum
ax = axes[0]
labels = list(result.cost_breakdown.keys())
values = list(result.cost_breakdown.values())
ax.pie(values, labels=[l.split("(")[0].strip() for l in labels],
       autopct="%1.1f%%", startangle=90)
ax.set_title("Cost breakdown at optimum")

# Right: stacked area of cost components over evaluations
ax = axes[1]
evals = range(1, len(result.eval_history) + 1)
ax.fill_between(evals, 0, cost_components["moisture"], alpha=0.6, label="Moisture")
ax.fill_between(evals, cost_components["moisture"],
                np.array(cost_components["moisture"]) + np.array(cost_components["length"]),
                alpha=0.6, label="Length")
ax.fill_between(evals,
                np.array(cost_components["moisture"]) + np.array(cost_components["length"]),
                np.array(cost_components["moisture"]) + np.array(cost_components["length"]) + np.array(cost_components["energy"]),
                alpha=0.6, label="Energy")
ax.set_xlabel("Function evaluation #")
ax.set_ylabel("Weighted cost")
ax.set_title("Cost components over evaluations")
ax.legend()
ax.grid(True, ls=":", alpha=0.4)

plt.tight_layout()

## 10. Decision Variable Exploration

Scatter plots of where the optimizer sampled in the 3D decision space,
coloured by cost. Shows exploration vs exploitation behaviour.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

pairs = [
    ("length", "temp", "Chamber length [m]", "Chamber temp [°C]"),
    ("length", "airflow", "Chamber length [m]", "Airflow [m/s]"),
    ("temp", "airflow", "Chamber temp [°C]", "Airflow [m/s]"),
]

costs = df_hist["cost"]
vmin, vmax = costs.quantile(0.01), costs.quantile(0.95)

for ax, (xk, yk, xl, yl) in zip(axes, pairs):
    sc = ax.scatter(
        df_hist[xk], df_hist[yk],
        c=costs, cmap="viridis", s=12, alpha=0.6,
        vmin=vmin, vmax=vmax,
    )
    # Mark optimum
    opt_vals = {"length": result.optimal_length, "temp": result.optimal_temp, "airflow": result.optimal_airflow}
    ax.scatter(opt_vals[xk], opt_vals[yk], marker="*", s=200, c="red", edgecolors="k", zorder=5, label="Optimum")
    ax.set_xlabel(xl)
    ax.set_ylabel(yl)
    ax.legend(fontsize=8)
    ax.grid(True, ls=":", alpha=0.3)

fig.colorbar(sc, ax=axes, label="Cost", shrink=0.8)
fig.suptitle("Optimizer sampling in decision space", y=1.02)


## 11. Radial Moisture Profile at Optimum

The radial moisture distribution across the filament cross-section
at the exit of the optimized dryer — shows how deeply drying penetrates.

In [ ]:
sim = result.simulation
diff = sim.diffusion
r_um = diff.r * 1e6  # m → µm

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: radial profile at exit
ax = axes[0]
ax.plot(r_um, diff.C[:, -1] * 100, "b-", lw=2, label="Exit")
ax.plot(r_um, diff.C[:, 0] * 100, "k--", lw=1, alpha=0.5, label="Initial")
ax.axhline(sim.final_moisture * 100, color="r", ls=":", lw=1, label=f"Avg: {sim.final_moisture*100:.3f} wt%")
target = config.effective_target_moisture()
if target is not None:
    ax.axhline(target * 100, color="orange", ls="-.", lw=1.5, label=f"Max print: {target*100:.2f} wt%")
ax.set_xlabel("Radius [µm]")
ax.set_ylabel("Moisture [wt%]")
ax.set_title("Radial moisture profile at dryer exit")
ax.legend()
ax.grid(True, ls=":", alpha=0.4)

# Right: volume-averaged moisture over time
ax = axes[1]
ax.plot(diff.t, diff.C_avg * 100, "b-", lw=2)
ax.axhline(sim.initial_moisture * 100, color="k", ls="--", lw=1, alpha=0.5, label="Initial")
if target is not None:
    ax.axhline(target * 100, color="orange", ls="-.", lw=1.5, label=f"Max print: {target*100:.2f} wt%")
ax.set_xlabel("Time [s]")
ax.set_ylabel("Volume-averaged moisture [wt%]")
ax.set_title("Moisture evolution during transit")
ax.legend()
ax.grid(True, ls=":", alpha=0.4)

plt.tight_layout()

## 12. Multi-Material Comparison

Run the optimizer for all materials and compare optimal designs.

In [ ]:
initial_moistures = {
    "PA6": 0.04, "PA12": 0.01, "PETG": 0.004, "PLA": 0.005,
    "TPU": 0.008, "PVA": 0.06, "ABS": 0.003,
    "PC": 0.002, "ASA": 0.004, "PPA": 0.02,
}

multi_results = {}
for key, mat in MATERIALS.items():
    fil = FilamentConfig(material=mat, initial_moisture=initial_moistures[key], flow_rate=8.0)
    cfg = OptimizationConfig(filament=fil)
    r = optimize(cfg, num_evaluations=NUM_EVALUATIONS)
    multi_results[key] = r
    target = cfg.effective_target_moisture()
    target_str = f"  target={target*100:.2f}%" if target else ""
    met = "✓" if r.simulation.final_moisture <= target else "✗" if target else ""
    print(f"{mat.name:25s}  cost={r.optimal_cost:.6f}  L={r.optimal_length*1e3:.0f}mm  T={r.optimal_temp:.1f}°C  v={r.optimal_airflow:.2f}m/s{target_str} {met}  ({r.wall_time:.1f}s)")

# Summary table
rows = []
for key, r in multi_results.items():
    target = r.config.effective_target_moisture()
    rows.append({
        "Material": MATERIALS[key].name,
        "Length [mm]": f"{r.optimal_length * 1e3:.0f}",
        "Temp [°C]": f"{r.optimal_temp:.1f}",
        "Airflow [m/s]": f"{r.optimal_airflow:.2f}",
        "Final moisture [wt%]": f"{r.simulation.final_moisture * 100:.3f}",
        "Threshold [wt%]": f"{target * 100:.2f}" if target else "—",
        "Meets threshold": "✓" if target and r.simulation.final_moisture <= target else "✗" if target else "—",
        "Efficiency [%]": f"{r.simulation.drying_efficiency * 100:.1f}",
        "Cost": f"{r.optimal_cost:.4f}",
        "Wall time [s]": f"{r.wall_time:.1f}",
    })
pd.DataFrame(rows)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

mat_names = [MATERIALS[k].name for k in multi_results]
x = np.arange(len(mat_names))
w = 0.6

# Chamber length
ax = axes[0, 0]
lengths = [multi_results[k].optimal_length * 1e3 for k in multi_results]
ax.bar(x, lengths, w, color="steelblue")
ax.set_xticks(x); ax.set_xticklabels(mat_names, rotation=30, ha="right")
ax.set_ylabel("Chamber length [mm]")
ax.set_title("Optimal chamber length by material")
ax.grid(True, axis="y", ls=":", alpha=0.4)

# Temperature
ax = axes[0, 1]
temps = [multi_results[k].optimal_temp for k in multi_results]
max_temps = [MATERIALS[k].max_temp for k in multi_results]
ax.bar(x - 0.15, temps, 0.3, label="Optimal", color="coral")
ax.bar(x + 0.15, max_temps, 0.3, label="Max safe", color="lightgray", edgecolor="k")
ax.set_xticks(x); ax.set_xticklabels(mat_names, rotation=30, ha="right")
ax.set_ylabel("Temperature [°C]")
ax.set_title("Optimal vs max safe temperature")
ax.legend()
ax.grid(True, axis="y", ls=":", alpha=0.4)

# Drying efficiency
ax = axes[1, 0]
effs = [multi_results[k].simulation.drying_efficiency * 100 for k in multi_results]
ax.bar(x, effs, w, color="seagreen")
ax.set_xticks(x); ax.set_xticklabels(mat_names, rotation=30, ha="right")
ax.set_ylabel("Drying efficiency [%]")
ax.set_title("Drying efficiency at optimum")
ax.grid(True, axis="y", ls=":", alpha=0.4)

# Wall time
ax = axes[1, 1]
times = [multi_results[k].wall_time for k in multi_results]
ax.bar(x, times, w, color="orchid")
ax.set_xticks(x); ax.set_xticklabels(mat_names, rotation=30, ha="right")
ax.set_ylabel("Wall time [s]")
ax.set_title("Optimization time by material")
ax.grid(True, axis="y", ls=":", alpha=0.4)

plt.tight_layout()

## 13. Convergence Comparison Across Materials

Overlay the best-so-far cost curves to see which materials are
harder for the optimizer.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for key, r in multi_results.items():
    costs = pd.Series([e["cost"] for e in r.eval_history])
    best = costs.cummin()
    # Normalize: fraction of initial best remaining
    best_norm = best / best.iloc[0]
    ax.plot(range(1, len(best_norm) + 1), best_norm, lw=1.5, label=MATERIALS[key].name)

ax.set_xlabel("Function evaluation #")
ax.set_ylabel("Normalized best cost (fraction of initial)")
ax.set_title("Convergence comparison across materials")
ax.set_yscale("log")
ax.legend()
ax.grid(True, ls=":", alpha=0.4)
plt.tight_layout()